In [1]:
import io
import time
import datetime as dt

import requests
import pandas as pd
import numpy as np

In [2]:
"""
Download the ECB euro area AAA-rated government bond zero-coupon (spot rate)
yield curve, daily observations, from 2000-01-01 to today, for a specific set
of maturities.

Source: ECB Statistical Data Warehouse / ECB Data Portal, SDMX REST API.
Page:   https://www.ecb.europa.eu/stats/financial_markets_and_interest_rates/euro_area_yield_curves/html/index.en.html

Dataflow: YC (Financial market data - yield curve)
Series key pattern:  B.U2.EUR.4F.G_N_A.SV_C_YM.SR_<maturity>
  B       = daily frequency (business week)
  U2      = euro area (changing composition)
  EUR     = euro
  4F      = ECB as financial market data provider
  G_N_A   = government bonds, nominal, AAA-rated issuers only
            (use G_N_C instead for the "all ratings" curve)
  SV_C_YM = Svensson model, continuous compounding, yield-error minimisation
  SR_x    = spot rate at maturity x  (e.g. SR_3M, SR_2Y, SR_10Y ...)

IMPORTANT: the ECB only started publishing this daily curve on 6 September
2004. Requesting from 2000-01-01 is harmless, the API simply returns
whatever exists in that window.

Also note: not every "month" code (4M, 5M, 7M, 8M, 10M, 11M) is guaranteed to
exist as a published series. The script requests each maturity individually
and simply skips (with a printed warning) any code the API rejects or returns
no data for, so a missing code will not break the rest of the download.
"""
# --------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------

BASE_URL = "https://data-api.ecb.europa.eu/service/data/YC/"
START_DATE = "2000-01-01"
END_DATE = dt.date.today().isoformat()

RATING = "G_N_A"          # AAA-rated government bonds
MODEL = "SV_C_YM"         # Svensson, continuous compounding, yield-error minimisation
FREQ_AREA_CCY_PROV = "B.U2.EUR.4F"

OUTPUT_CSV = f"ecb_aaa_spot_yield_curve_{END_DATE}.csv"

# --------------------------------------------------------------------------
# Build the list of requested maturities
# --------------------------------------------------------------------------

# 3M -> 1Y : every month
short_end = [f"{m}M" for m in range(3, 12)] + ["1Y"]          # 3M..11M, 1Y

# 1Y -> 10Y : full year terms only
long_end_annual = [f"{y}Y" for y in range(1, 11)]              # 1Y..10Y

# additional long maturities
extra = ["12Y", "15Y", "18Y", "20Y", "25Y", "30Y"]

# de-duplicate (1Y appears in both short_end and long_end_annual) keeping order
maturities = list(dict.fromkeys(short_end + long_end_annual + extra))

print(f"Requesting {len(maturities)} maturities:")
print(", ".join(maturities))
print()

Requesting 25 maturities:
3M, 4M, 5M, 6M, 7M, 8M, 9M, 10M, 11M, 1Y, 2Y, 3Y, 4Y, 5Y, 6Y, 7Y, 8Y, 9Y, 10Y, 12Y, 15Y, 18Y, 20Y, 25Y, 30Y



In [3]:
# --------------------------------------------------------------------------
# Fetch helper
# --------------------------------------------------------------------------


def fetch_maturity(maturity: str, session: requests.Session) -> pd.Series | None:
    """Fetch a single spot-rate maturity series as a date-indexed pandas Series."""
    series_key = f"{FREQ_AREA_CCY_PROV}.{RATING}.{MODEL}.SR_{maturity}"
    url = f"{BASE_URL}{series_key}"
    params = {
        "startPeriod": START_DATE,
        "endPeriod": END_DATE,
        "format": "csvdata",
    }

    try:
        r = session.get(url, params=params, timeout=120)
    except requests.RequestException as exc:
        print(f"  [skip] {maturity}: request failed ({exc})")
        return None

    if r.status_code != 200 or not r.text.strip():
        print(f"  [skip] {maturity}: HTTP {r.status_code} (series code may not exist)")
        return None

    try:
        df = pd.read_csv(io.StringIO(r.text))
    except Exception as exc:
        print(f"  [skip] {maturity}: could not parse response ({exc})")
        return None

    if df.empty or "TIME_PERIOD" not in df.columns or "OBS_VALUE" not in df.columns:
        print(f"  [skip] {maturity}: empty / unexpected response format")
        return None

    s = df.set_index("TIME_PERIOD")["OBS_VALUE"]
    s.index = pd.to_datetime(s.index)
    s = s.sort_index()
    s.name = maturity

    print(f"  [ok]   {maturity:<4}: {len(s):>5} obs "
          f"({s.index.min().date()} -> {s.index.max().date()})")
    return s

In [4]:
# --------------------------------------------------------------------------
# Download all series
# --------------------------------------------------------------------------

session = requests.Session()
session.headers.update({"User-Agent": "python-requests (ECB yield curve downloader)"})

series_list = []
for mat in maturities:
    s = fetch_maturity(mat, session)
    if s is not None:
        series_list.append(s)
    time.sleep(0.3) 

if not series_list:
    raise RuntimeError(
        "No data could be retrieved. Check your internet connection or "
        "https://data.ecb.europa.eu for service status."
    )

# --------------------------------------------------------------------------
# Assemble into one wide dataframe (rows = dates, columns = maturities)
# --------------------------------------------------------------------------

curve = pd.concat(series_list, axis=1).sort_index()


def _sort_key(m: str):
    return (0, int(m[:-1])) if m.endswith("M") else (1, int(m[:-1]))


curve = curve[sorted(curve.columns, key=_sort_key)]
curve.index.name = "Date"

curve.to_csv(OUTPUT_CSV)

print()
print(f"Saved {curve.shape[0]} daily rows x {curve.shape[1]} maturity columns "
      f"to '{OUTPUT_CSV}'")
print()
print(curve.tail())

  [ok]   3M  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   4M  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   5M  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   6M  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   7M  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   8M  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   9M  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   10M :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   11M :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   1Y  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   2Y  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   3Y  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   4Y  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   5Y  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   6Y  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   7Y  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   8Y  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   9Y  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   10Y :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   12Y

In [ ]:
# missing tenors
# skip this ceil if there are no missing tenors from previous ceil 3
"""
missing_tenors = ['15Y',
 '30Y']

missing_series_list = []

for mat in missing_tenors:
    missing = fetch_maturity(mat, session)
    if missing is not None:
        missing_series_list.append(missing)
    time.sleep(0.3) 


# --------------------------------------------------------------------------
# Merge externally-recovered missing maturities into the curve dataframe
# --------------------------------------------------------------------------

for s in missing_series_list:
    s.index = pd.to_datetime(s.index)   # make sure it aligns with curve's date index
    curve[s.name] = s                   # pandas aligns on the date index automatically

# re-sort columns: 1M..11M by month, then 1Y..30Y by year (matches the
# ordering used throughout the rest of the script)
def _sort_key(col: str):
    token = col.split("_")[0]
    return (0, int(token[:-1])) if token.endswith("M") else (1, int(token[:-1]))

curve = curve[sorted(curve.columns, key=_sort_key)]

print(f"Merged {len(missing_series_list)} recovered maturities: "
      f"{[s.name for s in missing_series_list]}")
print(f"curve now has {curve.shape[1]} columns: {list(curve.columns)}")
"""

### Interpolation of 1M, 2M tenors - Methodological note
The ECB estimates the AAA spot curve only from 3 months onward using the Svensson model.

Details of the Svensson model can be found here: https://www.ecb.europa.eu/stats/financial_markets_and_interest_rates/euro_area_yield_curves/shared/pdf/technical_notes.pdf

In the Svensson model the spot rate $z(TTM)$ is defined as:

$$
\begin{aligned}
z(TTM) =\;& \beta^0 
+ \beta^1 \left[
\frac{1 - e^{-TTM / \tau_1}}{TTM / \tau_1}
\right] \\
&+ \beta^2 \left[
\frac{1 - e^{-TTM / \tau_1}}{TTM / \tau_1}
- e^{-TTM / \tau_1}
\right] \\
&+ \beta^3 \left[
\frac{1 - e^{-TTM / \tau_2}}{TTM / \tau_2}
- e^{-TTM / \tau_2}
\right]
\end{aligned}
$$

where $TTM$ is the term to maturity; $\beta$ and $\tau$ are the Svensson model parameters.


The missing 1M and 2M tenors are calculated as model-implied rates from the daily Svensson parameters: BETA0–BETA3 and TAU1–TAU2.

$\beta$ and $\tau$ are published by ECB:
https://data.ecb.europa.eu/data/data-categories/financial-markets-and-interest-rates/euro-area-yield-curves/aaa-rated-government-bonds-yield-curve/parameters?layerType=AL


In [5]:
 def fetch_parameter(param_code: str, session: requests.Session) -> pd.Series | None:
    """Fetch a single Svensson parameter (BETA0/1/2/3, TAU1, TAU2) series."""
    series_key = f"{FREQ_AREA_CCY_PROV}.{RATING}.{MODEL}.{param_code}"
    url = f"{BASE_URL}{series_key}"
    params = {
        "startPeriod": START_DATE,
        "endPeriod": END_DATE,
        "format": "csvdata",
    }
 
    try:
        r = session.get(url, params=params, timeout=60)
    except requests.RequestException as exc:
        print(f"  [skip] {param_code}: request failed ({exc})")
        return None
 
    if r.status_code != 200 or not r.text.strip():
        print(f"  [skip] {param_code}: HTTP {r.status_code}")
        return None
 
    try:
        df = pd.read_csv(io.StringIO(r.text))
    except Exception as exc:
        print(f"  [skip] {param_code}: could not parse response ({exc})")
        return None
 
    if df.empty or "TIME_PERIOD" not in df.columns or "OBS_VALUE" not in df.columns:
        print(f"  [skip] {param_code}: empty / unexpected response format")
        return None
 
    s = df.set_index("TIME_PERIOD")["OBS_VALUE"]
    s.index = pd.to_datetime(s.index)
    s = s.sort_index()
    s.name = param_code
 
    print(f"  [ok]   {param_code:<6}: {len(s):>5} obs "
          f"({s.index.min().date()} -> {s.index.max().date()})")
    return s
 
 
def svensson_spot_rate(ttm_years, b0, b1, b2, b3, t1, t2):
    """Evaluate the Svensson zero-coupon spot rate z(TTM), per the ECB's
    functional form (technical notes, 'Model description' section)."""
    x1 = ttm_years / t1
    x2 = ttm_years / t2
    return (
        b0
        + b1 * (1 - np.exp(-x1)) / x1
        + b2 * ((1 - np.exp(-x1)) / x1 - np.exp(-x1))
        + b3 * ((1 - np.exp(-x2)) / x2 - np.exp(-x2))
    )
 

In [9]:
# --------------------------------------------------------------------------
# Download the Svensson parameters and compute implied 1M / 2M spot rates
# --------------------------------------------------------------------------
 
PARAM_CODES = ["BETA0", "BETA1", "BETA2", "BETA3", "TAU1", "TAU2"]
 
print("\nFetching Svensson model parameters...")
param_series = {}
for code in PARAM_CODES:
    s = fetch_parameter(code, session)
    if s is not None:
        param_series[code] = s
    time.sleep(0.3)
 
if len(param_series) < len(PARAM_CODES):
    missing = sorted(set(PARAM_CODES) - set(param_series))
    print(f"\nWARNING: could not retrieve parameter(s) {missing}; "
          f"skipping implied 1M/2M calculation.")
else:
    params = pd.concat(param_series.values(), axis=1)
    params.columns = list(param_series.keys())
    params = params.sort_index().dropna()  # need all 6 params on a given day
 
    for label, ttm in [("1M", 1 / 12), ("2M", 2 / 12)]:
        col = f"{label}"
        curve[col] = svensson_spot_rate(
            ttm,
            params["BETA0"], params["BETA1"], params["BETA2"],
            params["BETA3"], params["TAU1"], params["TAU2"],
        )
 
    print(f"\nComputed Svensson-implied 1M/2M rates for {len(params)} dates.")
 
    # quick sanity check: model-implied 3M vs the officially published SR_3M
    if "3M" in curve.columns:
        check = curve[["3M"]].copy()
        check["3M_svensson_implied"] = svensson_spot_rate(
            3 / 12,
            params["BETA0"], params["BETA1"], params["BETA2"],
            params["BETA3"], params["TAU1"], params["TAU2"],
        )
        diff = (check["3M"] - check["3M_svensson_implied"]).abs()
        print(f"Sanity check -- published SR_3M vs Svensson formula at TTM=3M: "
              f"max abs diff = {diff.max():.6f} pp, mean abs diff = {diff.mean():.6f} pp")


Fetching Svensson model parameters...
  [ok]   BETA0 :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   BETA1 :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   BETA2 :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   BETA3 :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   TAU1  :  5592 obs (2004-09-06 -> 2026-07-22)
  [ok]   TAU2  :  5592 obs (2004-09-06 -> 2026-07-22)

Computed Svensson-implied 1M/2M rates for 5592 dates.
Sanity check -- published SR_3M vs Svensson formula at TTM=3M: max abs diff = 0.008528 pp, mean abs diff = 0.000002 pp


In [7]:
# --------------------------------------------------------------------------
# Sort columns and save
# --------------------------------------------------------------------------
 
 
def _sort_key(col: str):
    token = col.split("_")[0]  # e.g. "1M_svensson_implied" -> "1M"
    return (0, int(token[:-1])) if token.endswith("M") else (1, int(token[:-1]))
 
 
curve = curve[sorted(curve.columns, key=_sort_key)]
curve.index.name = "Date"
 
curve.to_csv(OUTPUT_CSV)
 
print(curve.tail())


Saved 5592 daily rows x 27 maturity columns to 'ecb_aaa_spot_yield_curve_2026-07-24.csv'

                  1M        2M        3M        4M        5M        6M  \
Date                                                                     
2026-07-16  2.201755  2.255448  2.303982  2.347824  2.387403  2.423113   
2026-07-17  2.228231  2.278523  2.324044  2.365219  2.402439  2.436065   
2026-07-20  2.228721  2.281975  2.329993  2.373254  2.412200  2.447234   
2026-07-21  2.231527  2.285055  2.333569  2.377512  2.417293  2.453288   
2026-07-22  2.225921  2.282429  2.333712  2.380224  2.422382  2.460570   

                  7M        8M        9M       10M  ...        7Y        8Y  \
Date                                                ...                       
2026-07-16  2.455312  2.484332  2.510473  2.534012  ...  2.970496  3.040756   
2026-07-17  2.466424  2.493820  2.518530  2.540807  ...  2.954135  3.021209   
2026-07-20  2.478724  2.507009  2.532397  2.555170  ...  2.964649  3.03353

In [12]:
from pathlib import Path

cwd = Path.cwd()
print()
print(f"Saved {curve.shape[0]} daily rows x {curve.shape[1]} maturity columns "
      f"to '{OUTPUT_CSV}'")
print(f"Output folder: {cwd}")
print(f"Full path to file: {cwd / OUTPUT_CSV}")


Saved 5592 daily rows x 27 maturity columns to 'ecb_aaa_spot_yield_curve_2026-07-24.csv'
Output folder: /Users/dima
Full path to file: /Users/dima/ecb_aaa_spot_yield_curve_2026-07-24.csv


In [13]:
curve

,1M,2M,3M,4M,5M,6M,7M,8M,9M,10M,...,7Y,8Y,9Y,10Y,12Y,15Y,18Y,20Y,25Y,30Y
Date,,,,,,,,,,,,,,,,,,,,,
2004-09-06,1.976563,2.005252,2.034172,2.063283,2.092544,2.121920,2.151377,2.180882,2.210407,2.239924,...,3.828505,3.974939,4.100712,4.209220,4.385283,4.576354,4.710629,4.779293,4.904536,4.988680
2004-09-07,1.974783,2.007910,2.040893,2.073717,2.106364,2.138822,2.171075,2.203112,2.234922,2.266495,...,3.839294,3.981472,4.103839,4.209626,4.381744,4.569196,4.701299,4.768947,4.892459,4.975495
2004-09-08,1.973762,2.009217,2.044384,2.079252,2.113815,2.148064,2.181994,2.215598,2.248871,2.281809,...,3.863753,4.003922,4.124390,4.228419,4.397466,4.581290,4.710671,4.776881,4.897700,4.978894
2004-09-09,1.975555,2.006353,2.037111,2.067808,2.098422,2.128935,2.159329,2.189588,2.219696,2.249641,...,3.789113,3.931745,4.054972,4.161872,4.336578,4.527995,4.663603,4.733254,4.860711,4.946545
2004-09-10,1.984750,2.009500,2.034645,2.060137,2.085928,2.111976,2.138239,2.164681,2.191267,2.217962,...,3.740559,3.886214,4.011975,4.120981,4.298915,4.493569,4.631319,4.702037,4.831415,4.918530
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-07-16,2.201755,2.255448,2.303982,2.347824,2.387403,2.423113,2.455312,2.484332,2.510473,2.534012,...,2.970496,3.040756,3.110348,3.177343,3.299243,3.445957,3.549601,3.597306,3.655626,3.648858
2026-07-17,2.228231,2.278523,2.324044,2.365219,2.402439,2.436065,2.466424,2.493820,2.518530,2.540807,...,2.954135,3.021209,3.087844,3.152122,3.269270,3.410322,3.509724,3.555243,3.609870,3.601332
2026-07-20,2.228721,2.281975,2.329993,2.373254,2.412200,2.447234,2.478724,2.507009,2.532397,2.555170,...,2.964649,3.033535,3.101927,3.167846,3.287882,3.432360,3.534308,3.581130,3.637944,3.630443
